# 원판 탐지 가림 증강 통제실험

첫 베이스라인에서 원판 박스의 50%를 가리면, 신뢰도 0.10과 IoU 0.50을 함께 통과한 검증 프레임이 38장에서 25장으로 줄었습니다. 이번에는 학습 중 부분 가림을 추가했을 때 이 성능이 나아지는지 확인합니다. 가림 증강의 참고 자료는 [Random Erasing Data Augmentation](https://arxiv.org/abs/1708.04896)입니다.

기존 베이스라인과 같은 사람별 분할, 사전학습 `yolo26n.pt`, Mac 장치, Ultralytics 설정을 사용합니다. 바꾸는 조건은 학습 중 원판을 일부 가리는 변환 하나입니다. 박스마다 30% 확률로 적용하고, 박스 안쪽 면적의 25~40%를 회색으로 가립니다. 완전히 가려진 원판의 학습 라벨을 만들지 않기 위해 경계는 그대로 둡니다.

In [ ]:
from random import Random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from matplotlib.patches import Rectangle
from ultralytics import YOLO

from wlcv.config import ANNOTATIONS, DATASET, RUNS, SPLIT, VIDEOS
from wlcv.data.dataset import prepare_dataset
from wlcv.data.plate_labels import read_boxes
from wlcv.evaluation.occlusion_study import validation_occlusion_scores
from wlcv.training.plate_occlusion import apply_plate_occlusion, train_occlusion

baseline_dir = RUNS / "plate_baseline"
candidate_dir = RUNS / "plate_occlusion"

## 1. 학습 전에 가려진 예시 보기

학습 프레임 한 장을 골라 원본과 가린 모습을 비교합니다. 예시 셀에서는 변화를 볼 수 있게 확률을 1로 설정합니다. 실제 학습에서는 기본 모자이크·반전 등의 처리 이후 박스당 확률 0.30으로 적용합니다. 검증과 원본 파일에는 가림을 적용하지 않습니다.

In [ ]:
stem = "a01_sn_f_01_f13"
with Image.open(DATASET / "images" / "train" / f"{stem}.jpg") as image:
    frame = np.array(image.convert("RGB"))

boxes = read_boxes(DATASET / "labels" / "train" / f"{stem}.txt")
images = torch.from_numpy(frame.copy()).permute(2, 0, 1)[None].float() / 255
apply_plate_occlusion(images, [0] * len(boxes), boxes, Random(0), probability=1)

height, width = frame.shape[:2]
figure, axes = plt.subplots(1, 2, figsize=(7, 5), layout="constrained")
for axis, pixels, title in zip(axes, (frame, images[0].permute(1, 2, 0)), ("Original", "Training preview")):
    axis.imshow(pixels)
    for cx, cy, bw, bh in boxes:
        axis.add_patch(Rectangle(((cx - bw / 2) * width, (cy - bh / 2) * height),
                                 bw * width, bh * height, fill=False, edgecolor="lime"))
    axis.set_title(title)
    axis.axis("off")
plt.show()

## 2. 가림을 추가한 모델 학습

원판과 정답 박스를 검수한 뒤 이 셀에서 `run_training = True`로 바꾸면 시작합니다. 시작 가중치는 사전학습 `yolo26n.pt`이고, 100에폭을 완료한 기존 `best.pt`는 비교용으로 사용합니다. 결과를 다시 볼 때는 학습을 끄고 `candidate_dir`를 확인하세요.

In [ ]:
run_training = False

if run_training:
    data_yaml = prepare_dataset(VIDEOS, ANNOTATIONS / "plate_boxes", SPLIT, DATASET)
    candidate_dir = train_occlusion(data_yaml, baseline_dir)
print("가림 증강 실행 폴더", candidate_dir)

## 3. 같은 검증셋에서 비교

두 모델로 같은 원본 39장과 원판 박스 중앙 50%를 가린 같은 39장을 평가합니다. 신뢰도 0.10, IoU 0.50을 통과한 프레임 수를 비교합니다. IoU 중앙값은 낮은 점수의 매칭 후보도 포함하고, 정답에 대응한 후보가 없는 프레임은 계산에서 제외합니다. 원본에서 탐지 성능이 떨어지지 않는지 검증 mAP50–95도 함께 봅니다.

실제 가림 사례인 a03 2번은 별도 행으로 봅니다. 결과가 좋아져도 다른 사람의 실제 가림 영상에서 효과를 확인할 필요가 있습니다. 선택한 기준은 이 검증셋에서 정한 값으로 기록합니다.

In [ ]:
results = []
actual_occlusion = []
if (candidate_dir / "weights" / "best.pt").is_file():
    for label, folder in [("baseline", baseline_dir), ("occlusion", candidate_dir)]:
        model = YOLO(str(folder / "weights" / "best.pt"))
        summary, frames = validation_occlusion_scores(model, DATASET)
        epochs = pd.read_csv(folder / "results.csv", skipinitialspace=True)
        summary["best_val_mAP50-95"] = epochs["metrics/mAP50-95(B)"].max()
        results.append(summary.assign(model=label))
        actual = frames[(frames["video_id"] == "a03_sn_f_01") & (frames["frame_index"] == 2)]
        actual_occlusion.append(actual.assign(model=label))

    comparison = pd.concat(results).reset_index()
    display(comparison.round(3))
    display(pd.concat(actual_occlusion).round(3))
    axis = comparison.pivot(index="condition", columns="model", values="matched_at_threshold").plot.bar(rot=0)
    axis.set_ylabel("Matched frames (conf ≥ 0.10, IoU ≥ 0.50)")
    plt.show()
else:
    print("가림 증강 학습이 끝난 뒤 이 셀을 실행하세요.")